# INV-M365-CJ persona-action reachability certification v1

## Purpose
- Prove the `G1` persona reachability claim on the governed runtime surface.
- Freeze the planned-persona action fence claim required before `G2` may begin.

## Lemma Mapping
- `L85_m365_persona_action_reachability_certification_v1`

## Invariant Support
- `L85_m365_persona_action_reachability_certification_v1.yaml`

## Expected Results
- `59/59` canonical persona-id resolutions
- `59/59` display-name resolutions
- `59/59` persona state reachability
- `5/5` planned-persona action fences


In [ ]:
import os
from pathlib import Path

import yaml
from fastapi.testclient import TestClient

os.environ.setdefault('M365_ACTOR_HEADER_FALLBACK', '1')
os.environ.setdefault('JWT_REQUIRED', '0')

from ops_adapter.main import app

repo_root = Path('.').resolve()
registry = yaml.safe_load((repo_root / 'registry' / 'persona_registry_v2.yaml').read_text(encoding='utf-8'))
personas = registry['personas']
planned_personas = [persona_id for persona_id, payload in personas.items() if payload['status'] == 'planned']
client = TestClient(app)

canonical_total = 0
display_total = 0
state_total = 0
planned_fences = []

for persona_id, payload in personas.items():
    display_name = payload['display_name']
    r_canonical = client.get('/personas/resolve', params={'query': persona_id})
    r_display = client.get('/personas/resolve', params={'query': display_name})
    r_state = client.get(f'/personas/{persona_id}/state')
    assert r_canonical.status_code == 200
    assert r_display.status_code == 200
    assert r_state.status_code == 200
    assert r_canonical.json()['canonical_agent'] == persona_id
    assert r_display.json()['canonical_agent'] == persona_id
    canonical_total += 1
    display_total += 1
    state_total += 1

for persona_id in planned_personas:
    r_action = client.post(
        f'/actions/{persona_id}/health.overview',
        headers={'X-User-Email': 'admin@example.com'},
        json={'params': {}},
    )
    assert r_action.status_code == 403
    assert r_action.json()['detail'] == f'persona_inactive:{persona_id}'
    planned_fences.append(persona_id)

result = {
    'total_personas': len(personas),
    'active_personas': sum(1 for payload in personas.values() if payload['status'] == 'active'),
    'planned_personas': len(planned_personas),
    'canonical_resolve_total': canonical_total,
    'display_name_resolve_total': display_total,
    'state_endpoint_total': state_total,
    'planned_persona_fence_total': len(planned_fences),
    'planned_persona_fences': planned_fences,
}

assert result['total_personas'] == 59
assert result['active_personas'] == 54
assert result['planned_personas'] == 5
assert result['canonical_resolve_total'] == 59
assert result['display_name_resolve_total'] == 59
assert result['state_endpoint_total'] == 59
assert result['planned_persona_fence_total'] == 5
result
